<a href="https://colab.research.google.com/github/ahmdarwish/LLM-Agentic-Legal-Information-Retrieval/blob/main/code_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
kagglehub.login()
path = kagglehub.competition_download('llm-agentic-legal-information-retrieval')

In [ ]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer, util
import torch


base_path = path + '/'

print(f"Loading data from: {base_path}")

 2. Load Data
train = pd.read_csv(os.path.join(base_path, 'train.csv'))
test = pd.read_csv(os.path.join(base_path, 'test.csv'))
laws = pd.read_csv(os.path.join(base_path, 'laws_de.csv'))

courts = pd.read_csv(os.path.join(base_path, 'court_considerations.csv'), nrows=50000)

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

corpus_df = pd.concat([
    laws[['citation', 'text']].rename(columns={'text': 'content'}),
    courts[['citation', 'text']].rename(columns={'text': 'content'})
]).reset_index(drop=True)

corpus_embeddings = model.encode(corpus_df['content'][:15000].tolist(), convert_to_tensor=True)

def get_citation(query):
    query_emb = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, corpus_embeddings, top_k=1)
    return corpus_df.iloc[hits[0][0]['corpus_id']]['citation']

print("Processing test queries...")
test['predicted_citations'] = test['query'].apply(get_citation)

test[['query_id', 'predicted_citations']].to_csv('submission.csv', index=False)
print("✅ submission.csv created!")

In [ ]:
print(laws[['citation', 'text']].head())

test['query_length'] = test['query'].apply(lambda x: len(x.split()))
print(f"Average query length: {test['query_length'].mean():.2f} words")